추정 교통량 / 자동차 등록 대수

In [ ]:
import pandas as pd
traffic_df = pd.read_csv("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/교통량 데이터(2019-2021)/전국 추정교통량 데이터_2021.csv", encoding='utf-8')
car_df = pd.read_csv("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/교통량 데이터(2019-2021)/전국 자동차 등록 대수 현황 (2024).csv", encoding='utf-8')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import koreanize_matplotlib
import warnings

# 경고 무시 설정
warnings.filterwarnings("ignore")

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우에서는 'Malgun Gothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS에서는 'AppleGothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'NanumGothic'  # Ubuntu와 같은 리눅스에서는 'NanumGothic'을 사용합니다.
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:
car_df

In [ ]:
traffic_df

In [ ]:
car_df = car_df.dropna().reset_index()

In [ ]:
car_df = car_df.drop(columns = "시군구")

In [ ]:
car_df = car_df.drop(columns = "index")

In [ ]:
car_df

In [ ]:
car_df

In [ ]:
sido_mapping = {
    11000: '서울특별시',
    21000: '부산광역시',
    22000: '대구광역시',
    23000: '인천광역시',
    24000: '광주광역시',
    25000: '대전광역시',
    26000: '울산광역시',
    29000: '세종특별자치시',
    31000: '경기도',
    32000: '강원도',
    33000: '충청북도',
    34000: '충청남도',
    35000: '전라북도',
    36000: '전라남도',
    37000: '경상북도',
    38000: '경상남도',
    39000: '제주도'
}

# 시도 코드를 시도명으로 변환
traffic_df['시도명'] = traffic_df['시도 코드'].map(sido_mapping)

# 결과 출력
print(traffic_df)

In [ ]:
traffic_df["시도명"].isnull().sum()

In [ ]:
# 지역명 표준화
def standardize_region_name(region_name):
    mapping = {
        '서울특별시': '서울',
        '경기도': '경기',
        '경상남도': '경남',
        '경상북도': '경북',
        '광주광역시': '광주',
        '대구광역시': '대구',
        '대전광역시': '대전',
        '부산광역시': '부산',
        '세종특별자치시': '세종',
        '울산광역시': '울산',
        '인천광역시': '인천',
        '전라남도': '전남',
        '전북특별자치도': '전북',
        '제주특별자치도': '제주',
        '충청남도': '충남',
        '충청북도': '충북',
        "강원특별자치도" : "강원",
        "강원도" : "강원",
        "전라북도" : "전북",
        "제주도" : "제주"
    }
    return mapping.get(region_name, region_name)

# 데이터프레임에 지역명 표준화 적용
traffic_df['시도명'] = traffic_df['시도명'].apply(standardize_region_name)
car_df['시도명'] = car_df['시도명'].apply(standardize_region_name)

In [ ]:
car_df

In [ ]:
traffic_df

In [ ]:
# '시도명'과 '모든차량 추정교통량'만 포함한 새로운 데이터프레임 생성
result_df = traffic_df[['시도명', '승용차 추정교통량']]

# '시도명'으로 그룹화하여 '모든차량 추정교통량'의 합계를 계산
result_df = result_df.groupby('시도명', as_index=False).sum()

# 결과 출력
print(result_df)

In [ ]:
# '시도명' 컬럼을 기준으로 두 데이터프레임 병합
merge_df = pd.merge(result_df, car_df, on='시도명', how='inner')

# 결과 출력
print(merge_df)

In [ ]:
# '총계' 컬럼의 쉼표 제거 및 정수형으로 변환
merge_df['총계'] = merge_df['총계'].str.replace(',', '').astype(int)

# 전체 교통추정량과 전체 합계 계산
total_traffic_volume = merge_df['승용차 추정교통량'].sum()
total_sum = merge_df['총계'].sum()

# 지역별 교통추정량 / 전체 교통추정량 컬럼 추가
merge_df['교통추정량 비율'] = merge_df['승용차 추정교통량'] / total_traffic_volume

# 지역별 합계 / 전체 합계 컬럼 추가
merge_df['합계 비율'] = merge_df['총계'] / total_sum

# 결과 출력
merge_df

# merge_df["추정교통량 비율"] = merge_df["승용차 추정교통량"] / merge_df["총계"]

In [ ]:
# 시각화
plt.figure(figsize=(12, 8))

# 교통추정량 비율 시각화
plt.subplot(1, 2, 1)
plt.barh(merge_df['시도명'], merge_df['교통추정량 비율'], color='skyblue')
plt.xlabel('교통추정량 비율')
plt.title('지역별 교통추정량 비율')
plt.gca().invert_yaxis()  # 상위 데이터가 위쪽에 위치하도록

# 합계 비율 시각화
plt.subplot(1, 2, 2)
plt.barh(merge_df['시도명'], merge_df['합계 비율'], color='lightgreen')
plt.xlabel('합계 비율')
plt.title('지역별 합계 비율')
plt.gca().invert_yaxis()  # 상위 데이터가 위쪽에 위치하도록

plt.tight_layout()
plt.show()

In [ ]:
# 비율을 계산하여 1보다 큰지 작은지에 따라 값 설정
merge_df['비율'] = merge_df['교통추정량 비율'] / merge_df['합계 비율']

# 비율이 1보다 큰 경우는 양수, 작은 경우는 음수로 설정하여 방향 조정
merge_df['표시값'] = merge_df['비율'].apply(lambda x: x if x > 1 else -x)

# 시각화
plt.figure(figsize=(12, 8))

# 막대 그래프 시각화
plt.barh(merge_df['시도명'], merge_df['표시값'], color=['skyblue' if x > 0 else 'lightcoral' for x in merge_df['표시값']])
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('비율 (교통추정량 비율 / 합계 비율)')
plt.title('차량 대수 대비 추정 교통량 비율 시각화')
plt.gca().invert_yaxis()  # 상위 데이터가 위쪽에 위치하도록
plt.show()

In [ ]:
# 승용차 추정교통량 / 총계 계산
merge_df['승용차 비율'] = merge_df['승용차 추정교통량'] / merge_df['총계']

# 비율을 내림차순으로 정렬
merge_df_sorted = merge_df.sort_values(by='승용차 비율', ascending=False)

# 시각화
plt.figure(figsize=(12, 8))

# 막대 그래프 시각화
bars = plt.barh(merge_df_sorted['시도명'], merge_df_sorted['승용차 비율'], color='skyblue')

# 기준선 추가
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=0.8)  # 기준선을 0.5로 설정 (승용차 비율 50% 기준)

# 레이블과 제목 설정
plt.xlabel('승용차 비율')
plt.title('지역별 승용차 추정 교통량 비율')

# 막대 그래프의 레이블 추가 (비율 표시)
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.02, bar.get_y() + bar.get_height() / 2, f'{width:.2f}', va='center')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 순위별 색상을 생성하는 함수
def colors_from_ranks(values, palette_name):
    # 순위를 계산합니다 (작은 값부터 큰 값까지 0, 1, 2, ...)
    ranks = pd.Series(values).rank(method='min').astype(int) - 1
    # 최대 순위를 기준으로 색상을 생성
    palette = sns.color_palette(palette_name, len(values))
    # 순위를 색상에 매핑
    return np.array(palette).take(ranks, axis=0)

# 승용차 비율 계산
merge_df['승용차 비율'] = merge_df['승용차 추정교통량'] / merge_df['총계']

# 비율을 내림차순으로 정렬
merge_df_sorted = merge_df.sort_values(by='승용차 비율', ascending=True)

# 승용차 비율의 평균 계산
mean_value = merge_df_sorted['승용차 비율'].mean()

# 색상 설정 (YlOrRd 팔레트 사용)
colors = colors_from_ranks(merge_df_sorted['승용차 비율'], "YlOrRd")

# 시각화
plt.figure(figsize=(12, 8))

# 막대 그래프 시각화 (막대마다 색상 다르게, 수직 막대 그래프)
bars = plt.bar(merge_df_sorted['시도명'], merge_df_sorted['승용차 비율'], color=colors)

# 평균선 추가 (점선으로 변경)
plt.axhline(y=mean_value, color='red', linestyle='--', linewidth=2, label=f'평균: {mean_value:.2f}')

# 평균 값을 그래프에 텍스트로 표시
plt.text(len(merge_df_sorted) - 1, mean_value + 0.02, f'평균: {mean_value:.2f}', color='red', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 레이블과 제목 설정
plt.ylabel('승용차 비율')
plt.xlabel('지역명')
plt.title('지역별 승용차 추정 교통량 비율')

# 막대 그래프의 레이블 추가 (비율 표시)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}', ha='center', va='bottom')

# x축 레이블 회전
plt.xticks(rotation=45)

# 범례 추가
plt.legend()

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 색상 팔레트 설정: 색상 리버스
def colors_from_values(values, base_color):
    # Normalize the values to range [0, 1]
    normalized = (values - min(values)) / (max(values) - min(values))
    # Create a color palette from the base color and reverse it
    palette = sns.light_palette(base_color, n_colors=len(values))[::-1]  # Reverse the palette
    return np.array(palette)

# 승용차 비율 계산
merge_df['승용차 비율'] = merge_df['승용차 추정교통량'] / merge_df['총계']

# 비율을 내림차순으로 정렬
merge_df_sorted = merge_df.sort_values(by='승용차 비율', ascending=False)

# 승용차 비율의 평균 계산
mean_value = merge_df_sorted['승용차 비율'].mean()

# 색상 설정 (Reversed palette with shades of "#00a4ef")
colors = colors_from_values(merge_df_sorted['승용차 비율'], "#00a4ef")

# 시각화
plt.figure(figsize=(12, 8))

# 막대 그래프 시각화 (막대마다 색상 다르게, 수직 막대 그래프)
bars = plt.bar(merge_df_sorted['시도명'], merge_df_sorted['승용차 비율'], color=colors)

# 평균선 추가 (점선으로 변경)
plt.axhline(y=mean_value, color='darkblue', linestyle='--', linewidth=2, label=f'평균: {mean_value:.2f}')

# 평균 값을 그래프에 텍스트로 표시
plt.text(len(merge_df_sorted) - 1, mean_value + 0.02, f'평균: {mean_value:.2f}', color='darkblue', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 레이블과 제목 설정
plt.ylabel('승용차 비율')
plt.xlabel('지역명')
plt.title('지역별 승용차 추정 교통량 비율')

# 막대 그래프의 레이블 추가 (비율 표시)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}', ha='center', va='bottom')

# x축 레이블 회전
plt.xticks(rotation=45)

# 범례 추가
plt.legend()

plt.show()
